# 00R · Corrida de referencia con el código R del autor

**Propósito:** ejecutar **sin modificar** los scripts R que compartió el autor (`recursos/01_mfNP.R` → `04_AlgScreening.R`) para tener:
1. **las métricas exactas del código original** (Tabla 2 y Tablas S1/S2), para compararlas con la publicación y con la reimplementación en Python;
2. **las filas exactas** de train/test y de cada fold que genera `set.seed(1987)` en R. La Parte B las usa para separar el efecto del generador aleatorio del efecto de la implementación.

Este notebook es **solo de referencia**: la reproducción entregada es la implementación propia en Python (`Parte_B_reproduccion.ipynb`). La corrida R también es la vía para validar lo que no se reimplementó en Python (bagged MARS, bagged MLP, la rejilla completa del RF). Ver `docs/supuestos_y_validaciones.md`.

## Requisitos (una sola vez)
1. Instalar R (≥ 4.3) desde <https://cran.r-project.org/bin/windows/base/>.
2. En una consola de R, instalar los paquetes del autor y el kernel de Jupyter (la celda 2 lista los que faltan):
   ```r
   install.packages(c("IRkernel", "tidymodels", "baguette", "earth", "nnet", "doParallel", "glmnet",
                      "kernlab", "kknn", "rpart", "randomForest", "probably", "poissonreg", "xgboost",
                      "readODS", "readxl", "peakRAM", "viridis", "ggsci", "cowplot", "patchwork",
                      "GGally", "ggfortify", "rgl", "plot3D", "xtable", "kernelshap", "shapviz"))
   IRkernel::installspec()
   ```
3. Abrir este notebook con el kernel **R**.

> ⏱ **Tiempo:** el autor midió ~17–20 min para la rejilla completa de la Tabla 2 (SI, Tabla S6). Con 6 procesos se estima ≈ 40 min – 1.5 h en total, incluido el screening de 13 algoritmos. Los tiempos reales de esta corrida se guardan en `resultados/referencia_R/tiempos_R.csv`.
>
> **No se reduce la rejilla del autor:** cambiaría 4 de las 6 particiones (ver `docs/reproduccion_paper.md` §3.5). Solo se limita el número de procesos, por memoria.
>
> Para correrlo sin interfaz:
> `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 notebooks/00R_referencia_autor.ipynb`

In [ ]:
# 1. Carpeta de trabajo: copia de recursos/ para no modificar los archivos originales
#    (los scripts del autor escriben en data/ y extra/)
raiz <- if (basename(getwd()) == "notebooks") normalizePath("..") else getwd()
ref <- file.path(raiz, "resultados", "referencia_R")
trabajo <- file.path(ref, "workdir")
unlink(trabajo, recursive = TRUE)
dir.create(trabajo, recursive = TRUE, showWarnings = FALSE)
invisible(file.copy(list.files(file.path(raiz, "recursos"), full.names = TRUE), trabajo, recursive = TRUE))
setwd(trabajo)
cat("Directorio de trabajo:", getwd(), "\n")
print(list.files())
sessionInfo()$R.version$version.string

### Versiones de `tune` / `tidymodels` iguales a las de la época del autor

El código del autor activa el paralelismo con `foreach` (`registerDoParallel`). **`tune` 2.x eliminó el soporte de `foreach`**, así que todo correría en un solo núcleo. Además, cambió la forma de generar las semillas: *"There will be a lack of reproducibility between objects created in this version of tune and previous versions"* (NEWS de tune 2.0.0).

Por eso se usan **`tune` 1.3.0 y `tidymodels` 1.3.0** (21-feb-2025, las últimas versiones compatibles con `foreach`, de la época del paper), instaladas en una biblioteca aparte: `%LOCALAPPDATA%/R/lib-tune-1.3.0`. `R_LIBS` se exporta para que los procesos paralelos carguen las mismas versiones. Detalle en `docs/reproduccion_paper.md` §3.6.

In [ ]:
lib_tune13 <- file.path(Sys.getenv("LOCALAPPDATA"), "R", "lib-tune-1.3.0")
stopifnot(dir.exists(file.path(lib_tune13, "tune")), dir.exists(file.path(lib_tune13, "tidymodels")))
.libPaths(c(lib_tune13, .libPaths()))
Sys.setenv(R_LIBS = paste(.libPaths(), collapse = ";"))   # los procesos paralelos heredan este orden
cat("tune:", as.character(packageVersion("tune")), "| tidymodels:", as.character(packageVersion("tidymodels")), "\n")
stopifnot(packageVersion("tune") == "1.3.0")

In [ ]:
# 2. Paquetes que usa el código del autor
requeridos <- c("readODS", "dplyr", "tidyr", "readxl", "peakRAM", "ggplot2", "viridis", "ggsci", "cowplot",
                "patchwork", "scales", "GGally", "ggfortify", "rgl", "plot3D", "xtable", "kernelshap",
                "shapviz", "tidymodels", "baguette", "earth", "doParallel", "glmnet", "kernlab", "kknn",
                "rpart", "randomForest", "probably", "poissonreg", "xgboost", "nnet")
faltan <- requeridos[!vapply(requeridos, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltan) > 0) {
  stop("Faltan paquetes: ", paste(faltan, collapse = ", "),
       "\nInstalarlos con install.packages(c(\"", paste(faltan, collapse = "\", \""), "\"))")
}
cat("✔ Todos los paquetes están instalados.\n")

## 3. Preparación de datos y Tabla 2 (scripts 01, 02 y 03, sin modificar)

In [ ]:
options(rgl.useNULL = TRUE)    # 01_mfNP.R carga rgl; sin ventana gráfica en ejecución sin interfaz
tiempos <- data.frame(etapa = character(), segundos = numeric())
marcar <- function(etapa, t0) {
  s <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
  tiempos[nrow(tiempos) + 1, ] <<- list(etapa, s)
  cat(sprintf("%s: %.1f min\n", etapa, s / 60))
}
t0 <- Sys.time()
source("01_mfNP.R")            # lee data/mfNP_database.csv, prep_main, ml_prep → dfML_lipid, dfML_PNP
marcar("01_mfNP.R", t0)
cat("dfML_lipid:", dim(dfML_lipid), "| dfML_PNP:", dim(dfML_PNP), "\n")
cat("Núcleos que registró el autor (80 %):", ncores, "\n")

### Límite de procesos (memoria)

`01_mfNP.R` registra el 80 % de los núcleos (`registerDoParallel(cores = ncores)`). Aquí se vuelve a registrar el paralelismo con **6 procesos**, sin modificar los scripts del autor. El número de procesos no cambia las particiones, porque estas se crean en la sesión principal.

In [ ]:
N_PROCESOS <- 6
doParallel::stopImplicitCluster()
doParallel::registerDoParallel(cores = N_PROCESOS)
cat("Procesos registrados (foreach):", foreach::getDoParWorkers(), "\n")
cat("Versión de tune:", as.character(packageVersion("tune")), "\n")

In [ ]:
t0 <- Sys.time()
source("02_rf_lipid_model.R")  # RF lípidos: rejilla 10x10x10, 10-fold CV, train/test
marcar("02_rf_lipid_model.R", t0)
t0 <- Sys.time()
source("03_rf_PLGA_model.R")   # RF PLGA
marcar("03_rf_PLGA_model.R", t0)

In [ ]:
# Exportar la Tabla 2 calculada por el código del autor
mejor <- function(b) if (is.null(b)) c(NA, NA, NA) else c(b$mtry, b$trees, b$min_n)
hp <- rbind(mejor(best_diam_lipid), mejor(best_pdi_lipid), mejor(best_ee_lipid),
            mejor(best_diam_pol), mejor(best_pdi_pol), mejor(best_ee_pol))
tabla2_R <- data.frame(
  familia = rep(c("lipidos", "plga"), each = 3),
  objetivo = rep(c("hdiam_nm", "pdi", "percEE"), 2),
  rmse_train = c(rmse_rf_lipid_diam_train, rmse_rf_lipid_pdi_train, rmse_rf_lipid_ee_train,
                 rmse_rf_pol_diam_train, rmse_rf_pol_pdi_train, rmse_rf_pol_ee_train),
  rmse_test = c(rmse_rf_lipid_diam_test, rmse_rf_lipid_pdi_test, rmse_rf_lipid_ee_test,
                rmse_rf_pol_diam_test, rmse_rf_pol_pdi_test, rmse_rf_pol_ee_test),
  rsq_train = c(rsq_rf_lipid_diam_train, rsq_rf_lipid_pdi_train, rsq_rf_lipid_ee_train,
                rsq_rf_pol_diam_train, rsq_rf_pol_pdi_train, rsq_rf_pol_ee_train),
  rsq_test = c(rsq_rf_lipid_diam_test, rsq_rf_lipid_pdi_test, rsq_rf_lipid_ee_test,
               rsq_rf_pol_diam_test, rsq_rf_pol_pdi_test, rsq_rf_pol_ee_test),
  mtry = hp[, 1], trees = hp[, 2], min_n = hp[, 3])
write.csv(tabla2_R, file.path(ref, "tabla2_rf_R.csv"), row.names = FALSE)
tabla2_R

In [ ]:
# Exportar las particiones exactas (train/test y fold de CV) con índices base 0 de dfML_*
exportar_particion <- function(split, folds, posiciones, familia, objetivo) {
  # posiciones: filas de dfML_* (base 1) que recibió initial_split (con el filtro de PdI si aplica)
  n <- length(posiciones)
  en_train <- split$in_id
  conjunto <- rep("test", n); conjunto[en_train] <- "train"
  fold <- rep(-1L, n)
  for (k in seq_along(folds$splits)) {
    evaluacion <- setdiff(seq_along(en_train), folds$splits[[k]]$in_id)   # posiciones dentro de training()
    fold[en_train[evaluacion]] <- k - 1L
  }
  data.frame(familia = familia, objetivo = objetivo, indice = posiciones - 1L,
             conjunto = conjunto, fold = fold)
}
todas <- seq_len(nrow(dfML_lipid)); filtro_l <- which(dfML_lipid$pdi <= pdi_lim)
todas_p <- seq_len(nrow(dfML_PNP));  filtro_p <- which(dfML_PNP$pdi <= pdi_lim)
particiones_R <- rbind(
  exportar_particion(diam_split_lipid, diam_folds_lipid, filtro_l, "lipidos", "hdiam_nm"),
  exportar_particion(pdi_split_lipid, pdi_folds_lipid, todas, "lipidos", "pdi"),
  exportar_particion(ee_split_lipid, ee_folds_lipid, todas, "lipidos", "percEE"),
  exportar_particion(diam_split_pol, diam_folds_pol, filtro_p, "plga", "hdiam_nm"),
  exportar_particion(pdi_split_pol, pdi_folds_pol, todas_p, "plga", "pdi"),
  exportar_particion(ee_split_pol, ee_folds_pol, todas_p, "plga", "percEE"))
write.csv(particiones_R, file.path(ref, "particiones_R.csv"), row.names = FALSE)
table(particiones_R$familia, particiones_R$objetivo, particiones_R$conjunto)

## 4. Comparación de 13 algoritmos (script 04, sin modificar)

In [ ]:
t0 <- Sys.time()
source("04_AlgScreening.R")
marcar("04_AlgScreening.R", t0)
write.csv(tiempos, file.path(ref, "tiempos_R.csv"), row.names = FALSE)
tiempos

In [ ]:
# Exportar las métricas de CV (sin redondear) con los nombres de algoritmo del SI
nombres_si <- c("bagged\nMARS" = "bagged MARS", "bagged\nMLP" = "bagged MLP", "bagged\ntrees" = "bagged trees",
                "xgboost" = "xgboost", "k-nearest\nneighbors" = "k-nearest neighbors", "LASSO" = "LASSO",
                "random\nforest" = "random forest", "RIDGE" = "RIDGE", "linear\nSVM" = "SVM, linear",
                "polynomial\nSVM" = "SVM, polynomial", "radial\nSVM" = "SVM, radial",
                "linear\nregression" = "linear regression", "Poisson\nregression" = "Poisson regression")
a_largo <- function(rmse, rsq, familia) {
  salida <- list()
  for (o in c("hdiam_nm", "pdi", "percEE")) {
    p <- c(hdiam_nm = "diam", pdi = "pdi", percEE = "ee")[[o]]
    salida[[o]] <- data.frame(familia = familia, objetivo = o, algoritmo = unname(nombres_si[rmse$algorithm]),
                              rmse = rmse[[paste0(p, "_rmse")]], rmse_se = rmse[[paste0(p, "_rmseSD")]],
                              rsq = rsq[[paste0(p, "_rsq")]], rsq_se = rsq[[paste0(p, "_rsqSD")]])
  }
  do.call(rbind, salida)
}
cv_R <- rbind(a_largo(df_lipid_varMods_rmse, df_lipid_varMods_rsq, "lipidos"),
              a_largo(df_pol_varMods_rmse, df_pol_varMods_rsq, "plga"))
write.csv(cv_R, file.path(ref, "cv_algoritmos_R.csv"), row.names = FALSE)
cv_R

## 5. Archivos generados

| Archivo | Contenido | Lo usa |
|---|---|---|
| `resultados/referencia_R/tabla2_rf_R.csv` | Tabla 2 calculada por el código del autor (+ hiperparámetros elegidos) | Parte B, sección 4 |
| `resultados/referencia_R/particiones_R.csv` | Filas de train/test y fold de cada modelo, generadas por `set.seed(1987)` en R | Parte B, sección 4 (variante "misma partición") |
| `resultados/referencia_R/cv_algoritmos_R.csv` | Métricas de CV de los 13 algoritmos, incluidos bagged MARS y bagged MLP | Parte B, sección 5 |
| `resultados/referencia_R/tiempos_R.csv` | Tiempo real de cada script | Documentación (`docs/reproduccion_paper.md` §3.3) |

Después de esta corrida, volver a ejecutar `Parte_B_reproduccion.ipynb` y actualizar `docs/supuestos_y_validaciones.md`.